<a href="https://colab.research.google.com/github/ADean03/SmartSpace-AI-powered-wall-decoration-placement/blob/main/testingstructure/src/SmartSpaceModelTrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!ls /content


datasets  drive  sample_data


In [ ]:
!mkdir -p /content/datasets


In [ ]:
src = "/content/drive/My Drive/SmartSpace/homeobjects-3K-masked"
dst = "/content/datasets/homeobjects-3K-masked"
!rsync -ah --progress "$src" "$dst"

Streaming output truncated to the last 5000 lines.
            659 100%    1.66kB/s    0:00:00 (xfr#5981, to-chk=2499/8487)
homeobjects-3K-masked/labels/train/living_room_1p (223)_dup.txt
            659 100%    0.81kB/s    0:00:00 (xfr#5982, to-chk=2498/8487)
homeobjects-3K-masked/labels/train/living_room_1p (223)_dup_flip.txt
            572 100%    0.38kB/s    0:00:01 (xfr#5983, to-chk=2497/8487)
homeobjects-3K-masked/labels/train/living_room_1p (224).txt
            438 100%    0.00kB/s    0:00:00 (xfr#5984, to-chk=2496/8487)
homeobjects-3K-masked/labels/train/living_room_1p (224)_dup.txt
            438 100%    1.53kB/s    0:00:00 (xfr#5985, to-chk=2495/8487)
homeobjects-3K-masked/labels/train/living_room_1p (224)_dup_flip.txt
            384 100%    0.55kB/s    0:00:00 (xfr#5986, to-chk=2494/8487)
homeobjects-3K-masked/labels/train/living_room_1p (225).txt
            482 100%    0.42kB/s    0:00:01 (xfr#5987, to-chk=2493/8487)
homeobjects-3K-masked/labels/train/living_room_1p (2

In [ ]:
import os
if os.path.exists(dst):
    print(f"✅ Dataset successfully copied to {dst}")
    print("Sample contents:")
    print(os.listdir(dst))
else:
    print("❌ Copy failed. Check the paths.")

✅ Dataset successfully copied to /content/datasets/homeobjects-3K-masked
Sample contents:
['homeobjects-3K-masked']


In [ ]:
yolo_data_path = os.path.join(dst, "HomeObjects-3K-masked.yaml")
print(f"Use this path in YOLOv8 training: {yolo_data_path}")

Use this path in YOLOv8 training: /content/datasets/homeobjects-3K-masked/HomeObjects-3K-masked.yaml


In [ ]:
!nvidia-smi


Wed Nov 19 04:22:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))


PyTorch CUDA available: True
Device name: Tesla T4


In [ ]:
import os

dst = "/content/datasets/homeobjects-3K-masked"
print("Contents of dataset folder:", os.listdir(dst))


Contents of dataset folder: ['homeobjects-3K-masked']


Updated model training

In [ ]:
# -----------------------------
# 1️⃣ Install / update ultralytics
# -----------------------------
!pip install -U ultralytics

In [ ]:
# -----------------------------
# 2️⃣ Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
# -----------------------------
# 3️⃣ Set dataset paths
# -----------------------------
import os
import yaml
from ultralytics import YOLO
from sklearn.model_selection import KFold

# Source & destination
src = "/content/drive/My Drive/SmartSpace/homeobjects-3K-masked"
dst = "/content/datasets/homeobjects-3K-masked"
os.makedirs("/content/datasets", exist_ok=True)

# Copy if not exists locally
if not os.path.exists(dst):
    !rsync -ah --progress "$src/" "$dst/"
else:
    print(f"✅ Dataset already exists locally at {dst}")

# Paths
images_dir = os.path.join(dst, "images")  # contains train/ val/
labels_dir = os.path.join(dst, "labels")  # contains train/ val/

# YAML path
yolo_data_path = os.path.join(dst, "homeobjects-3K-masked", "HomeObjects-3K-masked.yaml")
print("Use this path in YOLOv8 training:", yolo_data_path)

✅ Dataset already exists locally at /content/datasets/homeobjects-3K-masked
Use this path in YOLOv8 training: /content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/HomeObjects-3K-masked.yaml


In [ ]:
# -----------------------------
# 4️⃣ Load original YAML
# -----------------------------
with open(yolo_data_path) as f:
    data = yaml.safe_load(f)

train_images = data['train']  # list of image file paths
val_images = data['val']      # list of val image file paths


In [ ]:
import yaml
from sklearn.model_selection import KFold

# -----------------------------
# 5️⃣ Prepare k-fold on training set only
# -----------------------------
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Infer number of classes from names
nc = len(data['names'])

fold_num = 1
for train_idx, _ in kf.split(train_images):
    # Select training images for this fold
    fold_train = [train_images[i] for i in train_idx]

    # Keep original validation set
    fold_val = val_images

    # Create fold YAML using directories
    fold_yaml = {
        'train': fold_train,
        'val': fold_val,
        'nc': nc,
        'names': data['names']
    }

    fold_yaml_path = f"/content/datasets/fold_{fold_num}.yaml"
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f)

    print(f"\n✅ Fold {fold_num} YAML created at {fold_yaml_path}")
    fold_num += 1





✅ Fold 1 YAML created at /content/datasets/fold_1.yaml

✅ Fold 2 YAML created at /content/datasets/fold_2.yaml

✅ Fold 3 YAML created at /content/datasets/fold_3.yaml

✅ Fold 4 YAML created at /content/datasets/fold_4.yaml

✅ Fold 5 YAML created at /content/datasets/fold_5.yaml


In [ ]:
import yaml

with open(yolo_data_path) as f:
    data = yaml.safe_load(f)

print(data)


{'path': '/content/datasets/homeobjects/homeobjects-3K-masked', 'train': '/content/datasets/homeobjects/homeobjects-3K-masked/images/train', 'val': '/content/datasets/homeobjects/homeobjects-3K-masked/images/val', 'test': None, 'names': {0: 'bed', 1: 'sofa', 2: 'chair', 3: 'table', 4: 'lamp', 5: 'tv', 6: 'laptop', 7: 'wardrobe', 8: 'window', 9: 'door', 10: 'potted plant', 11: 'photo frame'}}


In [ ]:
# -----------------------------
# 6️⃣ Train YOLOv8 model on this fold
# -----------------------------
model = YOLO('yolov8n.pt')

results = model.train(
  data=fold_yaml_path,
  epochs=80,
  imgsz=640,
  batch=16,
  device=0,
  name=f'yolov8n_fold{fold_num}'
)

fold_num += 1

print("\n✅ All folds trained successfully!")

SyntaxError: '[31m[1mhyperparameter_evolve[0m' is not a valid YOLO argument. 

    Arguments received: ['yolo', '-f', '/root/.local/share/jupyter/runtime/kernel-ceddb47a-b25a-438f-8017-5ae95629183b.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['pose', 'segment', 'obb', 'detect', 'classify']
                MODE (required) is one of ['benchmark', 'train', 'export', 'predict', 'val', 'track']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo11n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo11n-seg.pt source='https://yo

In [ ]:
# 1️⃣ Make the target folder structure
!mkdir -p /content/datasets/homeobjects

# 2️⃣ Move your existing dataset from /content/your_dataset_folder to the target
!mv /content/your_dataset_folder /content/datasets/homeobjects/homeobjects-3K-masked


In [ ]:
# -----------------------------
# 1️⃣ Mount Google Drive (if not done)
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -----------------------------
# 2️⃣ Imports
# -----------------------------
import os
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO

# -----------------------------
# 3️⃣ Paths
# -----------------------------
dataset_dir = "/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked"
yolo_data_path = os.path.join(dataset_dir, "HomeObjects-3K-masked.yaml")

# -----------------------------
# 4️⃣ Load YAML
# -----------------------------
with open(yolo_data_path) as f:
    data = yaml.safe_load(f)

# YOLO expects 'nc' (number of classes)
nc = len(data['names'])

# Get train/val image paths
train_images = [os.path.join(dataset_dir, "images/train", fname)
                for fname in os.listdir(os.path.join(dataset_dir, "images/train"))
                if fname.endswith(('.jpg', '.png'))]

val_images = [os.path.join(dataset_dir, "images/val", fname)
              for fname in os.listdir(os.path.join(dataset_dir, "images/val"))
              if fname.endswith(('.jpg', '.png'))]

print(f"✅ Found {len(train_images)} train and {len(val_images)} val images.")

# -----------------------------
# 5️⃣ Prepare k-Fold YAMLs
# -----------------------------
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_num = 1

fold_yaml_paths = []

for train_idx, _ in kf.split(train_images):
    fold_train = [train_images[i] for i in train_idx]
    fold_val = val_images  # keep original val

    fold_yaml = {
        'train': fold_train,
        'val': fold_val,
        'nc': nc,
        'names': data['names']
    }

    fold_yaml_path = f"/content/datasets/fold_{fold_num}.yaml"
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f)

    fold_yaml_paths.append(fold_yaml_path)
    print(f"✅ Fold {fold_num} YAML created: {fold_yaml_path}")
    fold_num += 1

# -----------------------------
# 6️⃣ K-Fold Training with Hyperparameter Evolution
# -----------------------------
# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

for fold_num, fold_yaml_path in enumerate(fold_yaml_paths, 1):
    print(f"\n🔹 Training Fold {fold_num}...")

    # 1️⃣ Normal training
    results = model.train(
        data=fold_yaml_path,
        epochs=80,       # adjust based on dataset size
        imgsz=640,       # can reduce to 320 for testing
        batch=16,        # adjust based on GPU memory
        device=0,
        name=f'yolov8n_fold{fold_num}'
    )

    # 2️⃣ Hyperparameter evolution (optional)
    print(f"🔹 Running hyperparameter evolution for Fold {fold_num}...")
    model.hyperparameter_evolve(
        data=fold_yaml_path,
        generations=5,      # small number for Colab-friendly runs
        population=10,      # candidate sets per generation
        save_dir=f'yolov8n_fold{fold_num}_evolve'
    )

print("\n✅ All folds trained successfully!")


Mounted at /content/drive
✅ Found 3835 train and 404 val images.
✅ Fold 1 YAML created: /content/datasets/fold_1.yaml
✅ Fold 2 YAML created: /content/datasets/fold_2.yaml
✅ Fold 3 YAML created: /content/datasets/fold_3.yaml
✅ Fold 4 YAML created: /content/datasets/fold_4.yaml
✅ Fold 5 YAML created: /content/datasets/fold_5.yaml

🔹 Training Fold 1...
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/fold_1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, 

FileNotFoundError: [34m[1mtrain: [0mError loading data from ['/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1325.jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1669.jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_322.jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1058.jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1p (405).jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1960_dup_flip.jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1p (401).jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/living_room_1p (82).jpg', '/content/datasets/homeobjects-3K-masked/homeobjects-3K-masked/images/train/s (236).jpg', '/content/datasets/homeobjects-3K-m

In [ ]:
import shutil
import os

datasets_dir = "/content/datasets"

# Check if it exists
if os.path.exists(datasets_dir):
    print(f"⚠️ Deleting existing folder: {datasets_dir}")
    shutil.rmtree(datasets_dir)  # this deletes the folder and all its contents
else:
    print(f"✅ {datasets_dir} does not exist, nothing to delete.")

# Recreate an empty datasets folder
os.makedirs(datasets_dir, exist_ok=True)
print(f"✅ Empty folder created: {datasets_dir}")


⚠️ Deleting existing folder: /content/datasets
✅ Empty folder created: /content/datasets


In [ ]:
# -----------------------------
# 1️⃣ Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -----------------------------
# 2️⃣ Imports
# -----------------------------
import os
import yaml
import shutil
from sklearn.model_selection import KFold
from ultralytics import YOLO

# -----------------------------
# 3️⃣ Copy dataset from Drive to /content
# -----------------------------
src = "/content/drive/My Drive/SmartSpace/homeobjects-3K-masked"  # your Drive folder
dst = "/content/datasets/homeobjects/homeobjects-3K-masked"

os.makedirs("/content/datasets", exist_ok=True)

if not os.path.exists(dst):
    print(f"📦 Copying dataset from Drive to {dst} ...")
    shutil.copytree(src, dst, dirs_exist_ok=True)
else:
    print(f"✅ Dataset already exists locally at {dst}")

# -----------------------------
# 4️⃣ Load original YAML
# -----------------------------
dataset_dir = dst  # folder containing images/labels and YAML
yolo_yaml_file = os.path.join(dataset_dir, "HomeObjects-3K-masked.yaml")

with open(yolo_yaml_file) as f:
    data = yaml.safe_load(f)

nc = len(data['names'])  # number of classes

# Get train/val image paths
train_images = [os.path.join(dataset_dir, "images/train", fname)
                for fname in os.listdir(os.path.join(dataset_dir, "images/train"))
                if fname.lower().endswith(('.jpg', '.png'))]

val_images = [os.path.join(dataset_dir, "images/val", fname)
              for fname in os.listdir(os.path.join(dataset_dir, "images/val"))
              if fname.lower().endswith(('.jpg', '.png'))]

print(f"✅ Found {len(train_images)} train and {len(val_images)} val images.")

# -----------------------------
# 5️⃣ Prepare k-Fold YAMLs with train.txt / val.txt
# -----------------------------
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_yaml_paths = []

for fold_num, (train_idx, _) in enumerate(kf.split(train_images), 1):
    fold_train = [train_images[i] for i in train_idx]
    fold_val = val_images  # keep original val

    # Save train.txt / val.txt
    fold_train_txt = os.path.join(dataset_dir, f"fold_{fold_num}_train.txt")
    fold_val_txt   = os.path.join(dataset_dir, f"fold_{fold_num}_val.txt")

    with open(fold_train_txt, 'w') as f:
        f.write("\n".join(fold_train))
    with open(fold_val_txt, 'w') as f:
        f.write("\n".join(fold_val))

    # YAML for this fold
    fold_yaml = {
        'train': fold_train_txt,
        'val': fold_val_txt,
        'nc': nc,
        'names': data['names']
    }

    fold_yaml_path = os.path.join(dataset_dir, f"fold_{fold_num}.yaml")
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f)

    fold_yaml_paths.append(fold_yaml_path)
    print(f"✅ Fold {fold_num} YAML created: {fold_yaml_path}")

# -----------------------------
# 6️⃣ K-Fold Training with Hyperparameter Evolution
# -----------------------------
model = YOLO('yolov8n.pt')  # load pretrained YOLOv8n

for fold_num, fold_yaml_path in enumerate(fold_yaml_paths, 1):
    print(f"\n🔹 Training Fold {fold_num}...")

    # Normal training old 40, 640
    results = model.train(
        data=fold_yaml_path,
        epochs=80,
        imgsz=640,
        batch=16,
        device=0,
        name=f'yolov8n_fold{fold_num}'
    )


print("\n✅ All folds trained successfully!")


Mounted at /content/drive
📦 Copying dataset from Drive to /content/datasets/homeobjects/homeobjects-3K-masked ...
✅ Found 3835 train and 404 val images.
✅ Fold 1 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_1.yaml
✅ Fold 2 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_2.yaml
✅ Fold 3 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_3.yaml
✅ Fold 4 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_4.yaml
✅ Fold 5 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_5.yaml

🔹 Training Fold 1...
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/homeobjects/home

AttributeError: 'DetectionModel' object has no attribute 'hyperparameter_evolve'

In [ ]:
from ultralytics import YOLO
model2 = YOLO('/content/drive/My Drive/SmartSpace/testimages/best (2).pt')

In [ ]:
inference_results = model2.predict(
    source='/content/drive/My Drive/SmartSpace/testimages',
    classes=[11],   # photo frames
    imgsz=640,
    conf=0.001,
    save=True,      # saves results in /content/runs/detect/predict
    save_txt=False, # optional
    show=True       # display images in notebook
)



image 1/1 /content/drive/My Drive/SmartSpace/testimages/testimageformodel1.jpg: 480x640 1 photo frame, 7.4ms
Speed: 3.9ms preprocess, 7.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /content/runs/detect/predict6


In [ ]:
from ultralytics import YOLO
model2 = YOLO('/content/drive/My Drive/SmartSpace/testimages/best (2).pt')
inference_results = model2.predict(
    source='/content/drive/My Drive/SmartSpace/testimages/testimageformodel2.jpg',
    classes=[11],   # photo frames
    imgsz=640,
    conf=0.0005,
    save=True,
    save_txt=False, # optional
    show=True       # display images in notebook
)


image 1/1 /content/drive/My Drive/SmartSpace/testimages/testimageformodel2.jpg: 480x640 1 photo frame, 11.9ms
Speed: 4.1ms preprocess, 11.9ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /content/runs/detect/predict14


In [ ]:
inference_results = model.predict(
    source='/content/datasets/homeobjects/homeobjects-3K-masked/images/val',
    classes=[11],   # photo frames
    imgsz=640,
    conf=0.25,
    save=True,      # saves results in /content/runs/detect/predict
    save_txt=False, # optional
    show=True       # display images in notebook
)


WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


image 1/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_10.jpg: 640x448 (no detections), 61.6ms
image 2/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1000.jpg: 640x640 (no detections), 20.0ms
image 3/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1004.jpg: 640x480 (no detections), 146.6ms
image 4/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1008.jpg: 448x640 1 photo frame, 40.0ms
image 5/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1012.jpg: 448x640 10 photo frames, 6.9ms
image 6/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1016.jpg: 448x640 1 photo frame, 6.0ms
image 7/404 /content/datasets/homeobjects/homeobjects-3K-masked/images/val/living_room_1020.jpg: 448x640 2 photo frames, 6.0ms
image 8/404 /content/datasets/h

In [ ]:
!pip install -U ultralytics
# -----------------------------
# 1️⃣ Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -----------------------------
# 2️⃣ Imports
# -----------------------------
import os
import yaml
import shutil
from sklearn.model_selection import KFold
from ultralytics import YOLO

# -----------------------------
# 3️⃣ Copy dataset from Drive to /content
# -----------------------------
src = "/content/drive/My Drive/SmartSpace/homeobjects-3K-masked"  # your Drive folder
dst = "/content/datasets/homeobjects/homeobjects-3K-masked"

os.makedirs("/content/datasets", exist_ok=True)

if not os.path.exists(dst):
    print(f"📦 Copying dataset from Drive to {dst} ...")
    shutil.copytree(src, dst, dirs_exist_ok=True)
else:
    print(f"✅ Dataset already exists locally at {dst}")

# -----------------------------
# 4️⃣ Load original YAML
# -----------------------------
dataset_dir = dst  # folder containing images/labels and YAML
yolo_yaml_file = os.path.join(dataset_dir, "HomeObjects-3K-masked.yaml")

with open(yolo_yaml_file) as f:
    data = yaml.safe_load(f)

nc = len(data['names'])  # number of classes

# Get train/val image paths
train_images = [os.path.join(dataset_dir, "images/train", fname)
                for fname in os.listdir(os.path.join(dataset_dir, "images/train"))
                if fname.lower().endswith(('.jpg', '.png'))]

val_images = [os.path.join(dataset_dir, "images/val", fname)
              for fname in os.listdir(os.path.join(dataset_dir, "images/val"))
              if fname.lower().endswith(('.jpg', '.png'))]

print(f"✅ Found {len(train_images)} train and {len(val_images)} val images.")

# -----------------------------
# 5️⃣ Prepare k-Fold YAMLs with train.txt / val.txt
# -----------------------------
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_yaml_paths = []

for fold_num, (train_idx, _) in enumerate(kf.split(train_images), 1):
    fold_train = [train_images[i] for i in train_idx]
    fold_val = val_images  # keep original val

    # Save train.txt / val.txt
    fold_train_txt = os.path.join(dataset_dir, f"fold_{fold_num}_train.txt")
    fold_val_txt   = os.path.join(dataset_dir, f"fold_{fold_num}_val.txt")

    with open(fold_train_txt, 'w') as f:
        f.write("\n".join(fold_train))
    with open(fold_val_txt, 'w') as f:
        f.write("\n".join(fold_val))

    # YAML for this fold
    fold_yaml = {
        'train': fold_train_txt,
        'val': fold_val_txt,
        'nc': nc,
        'names': data['names']
    }

    fold_yaml_path = os.path.join(dataset_dir, f"fold_{fold_num}.yaml")
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f)

    fold_yaml_paths.append(fold_yaml_path)
    print(f"✅ Fold {fold_num} YAML created: {fold_yaml_path}")

# -----------------------------
# 6️⃣ K-Fold Training with Hyperparameter Evolution
# -----------------------------
model = YOLO('yolov8n.pt')  # load pretrained YOLOv8n

for fold_num, fold_yaml_path in enumerate(fold_yaml_paths, 1):
    print(f"\n🔹 Training Fold {fold_num}...")

    # Normal training old 40, 640
    results = model.train(
        data=fold_yaml_path,
        epochs=40,
        imgsz=416,
        batch=16,
        device='cpu', #device=0
        name=f'yolov8n_fold{fold_num}'
    )


print("\n✅ All folds trained successfully!")


Mounted at /content/drive
✅ Dataset already exists locally at /content/datasets/homeobjects/homeobjects-3K-masked
✅ Found 3835 train and 404 val images.
✅ Fold 1 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_1.yaml
✅ Fold 2 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_2.yaml
✅ Fold 3 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_3.yaml
✅ Fold 4 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_4.yaml
✅ Fold 5 YAML created: /content/datasets/homeobjects/homeobjects-3K-masked/fold_5.yaml

🔹 Training Fold 1...
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/homeobjects/h

KeyboardInterrupt: 